In [ ]:
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from src.unit_proccessing import  *
from src.utils.stats_utils import *

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from pyod.models.thresholds import FILTER

In [ ]:
with initialize(config_path='../configuration', version_base='1.1'):
    config = compose(config_name='main.yaml')

In [ ]:
features_class = UnitDataProcessing(config)
self = features_class

In [ ]:
df = self.df_unit
self = features_class
feature_name = 'f__pause_count'
score_name = 's__pause_count'
df = df[(~pd.isnull(df[feature_name]))].copy()


|## Plot the distribution of answer time set

In [ ]:
df[feature_name].hist()

In [ ]:
df[score_name] = df[feature_name] / df['f__number_answered']
df[df[score_name]<2][score_name].hist()

In [ ]:
# Create a new column that has the hours mapped to order of frequency
sorted_pauses = df[feature_name].value_counts().index
hour_to_rank = {hour: rank for rank, hour in enumerate(sorted_pauses)}
# Sorting the DataFrame based on the 'frequency' column in descending order
df['frequency'] = df[feature_name].map(hour_to_rank)
df['frequency'].hist(bins=48)

## USE ECOD algorithm that makes use of cumulative function and is non-parametric for detecting anomalies in answer time set

In [ ]:
model = ECOD(contamination=0.31)#FILTER( random_state=42))

model.fit(df[['frequency']])

In [ ]:
df['anomaly'] = model.predict(df[['frequency']])

In [ ]:
df['anomaly'].value_counts(), df['anomaly'].value_counts()/df['anomaly'].count()

In [ ]:
c=60*100000#48*5
data_true = df[(df['anomaly']==0)&(df[feature_name]<c)][feature_name]
data_false = df[(df['anomaly']==1)&(df[feature_name]<c)][feature_name]

# Determine the bin edges based on the entire dataset
bins = np.histogram_bin_edges(df[(df[feature_name]<c)][feature_name], bins=48)


plt.hist(data_true, bins=bins, alpha=0.5, color='blue', label='True')
plt.hist(data_false, bins=bins, alpha=0.5, color='red', label='False')
plt.show()

In [ ]:
sns.boxplot(df,y=feature_name, x='anomaly')
plt.show()

In [ ]:
bins = np.histogram_bin_edges(df[df[feature_name]<df[df['anomaly']==0][feature_name].max()][feature_name], bins=48)
plt.hist(df[df[feature_name]<df[df['anomaly']==0][feature_name].max()][feature_name], bins=bins, alpha=0.5, color='blue', label='True')
plt.show()

In [ ]:
df.groupby('survey_version').anomaly.value_counts()

In [ ]:
df = df.groupby(['interview__id','survey_version'])['anomaly'].mean()
df = df.reset_index()

In [ ]:
df['survey_label'] = df['survey_version'].apply(lambda x: False if int(x.split('_')[2])<13 else True)

In [ ]:
sns.boxplot(df,y='anomaly', x='survey_label')
plt.show()